# 04 — Monitorización, logs centralizados y alertas automáticas

Este notebook explica Loki, Promtail, `system_events`, `alerts` y el servicio `automation`.

In [1]:

from pathlib import Path
import ast
import json
import subprocess
import textwrap
import re

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


# =============================================================================
# 0. Utilidades base del notebook
# =============================================================================

FENCE = "`" * 3


def md(text: str) -> None:
    """Renderiza Markdown desde una celda Python."""
    display(Markdown(textwrap.dedent(text).strip()))


def code_block(lang: str, text: str) -> None:
    """Renderiza bloques de código sin romper la celda Python."""
    display(Markdown(f"{FENCE}{lang}\n{textwrap.dedent(text).strip()}\n{FENCE}"))


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        if (candidate / "docker-compose.yml").exists():
            return candidate

    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_text(path: Path, default: str = "") -> str:
    """Lee texto de forma segura."""
    path = Path(path)

    if not path.exists():
        return default

    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return default


def read_json(path: Path, default=None):
    """Lee JSON de forma segura."""
    path = Path(path)

    if not path.exists():
        return default

    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"[error leyendo JSON] {path}: {exc}")
        return default


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta comando shell y devuelve stdout/stderr."""
    print(f"$ {cmd}")

    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )

        if result.stdout:
            print(result.stdout)

        if result.stderr:
            print(result.stderr)

        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")

    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def file_size_kb(path: Path) -> float | None:
    try:
        return round(path.stat().st_size / 1024, 2)
    except Exception:
        return None


def line_count(path: Path) -> int | None:
    try:
        return len(path.read_text(encoding="utf-8", errors="ignore").splitlines())
    except Exception:
        return None


def extract_public_symbols(path: Path) -> list[str]:
    """Extrae clases y funciones públicas de un archivo Python."""
    try:
        source = path.read_text(encoding="utf-8", errors="ignore")
        tree = ast.parse(source)

        functions = []
        classes = []

        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and not node.name.startswith("_"):
                functions.append(node.name)

            if isinstance(node, ast.ClassDef) and not node.name.startswith("_"):
                classes.append(node.name)

        out = [f"class {name}" for name in sorted(set(classes))]
        out += [f"def {name}" for name in sorted(set(functions))]
        return out

    except Exception:
        return []


def extract_compose_service_block(service_name: str) -> str:
    """Extrae aproximadamente el bloque de un servicio desde docker-compose.yml."""
    compose_path = ROOT / "docker-compose.yml"
    text = read_text(compose_path)

    if not text:
        return ""

    lines = text.splitlines()
    start_idx = None
    pattern = re.compile(rf"^\s{{2}}{re.escape(service_name)}:\s*$")

    for i, line in enumerate(lines):
        if pattern.match(line):
            start_idx = i
            break

    if start_idx is None:
        return ""

    block = [lines[start_idx]]

    for line in lines[start_idx + 1:]:
        if re.match(r"^\s{2}[a-zA-Z0-9_-]+:\s*$", line):
            break
        block.append(line)

    return "\n".join(block)


monitoring_root = ROOT / "monitoring"
automation_root = ROOT / "services" / "automation"
pipeline_root = ROOT / "services" / "pipeline"


# =============================================================================
# 1. Introducción
# =============================================================================

md("""
# 04 — Monitorización, calidad de datos y alertas

Este notebook explica la parte de monitorización y automatización operativa del proyecto.

El sistema cubre los tres requisitos del enunciado:

| Requisito | Implementación |
|---|---|
| Logging centralizado de procesos del pipeline | Logs JSON del pipeline recogidos por Promtail y enviados a Loki |
| Validación de calidad de datos | `validation_rules.yaml`, eventos `pipeline.validation.done` y colección `ingestion_rejects` |
| Alertas o notificaciones ante fallos | Servicio `automation` que lee `system_events` y crea documentos en `alerts` |

La idea principal es que el sistema no solo procese datos, sino que deje trazabilidad de lo que ha pasado.
""")


# =============================================================================
# 2. Componentes
# =============================================================================

md("""
## 1. Componentes de monitorización

Esta tabla resume qué papel tiene cada pieza.
""")

components_df = pd.DataFrame([
    {
        "componente": "logging_setup.py",
        "ubicación": "services/pipeline/app/logging_setup.py",
        "qué es": "Configuración de logs estructurados JSON.",
        "para qué sirve": "Hace que cada evento del pipeline salga por stdout con timestamp, level, service, correlation_id y event.",
        "dónde se ve": "docker compose logs pipeline; Loki si Promtail lo recoge.",
    },
    {
        "componente": "events.py",
        "ubicación": "services/pipeline/app/events.py",
        "qué es": "Emisor de eventos de dominio.",
        "para qué sirve": "Guarda eventos funcionales del pipeline en MongoDB, colección system_events.",
        "dónde se ve": "MongoDB: db.system_events.find().",
    },
    {
        "componente": "Loki",
        "ubicación": "monitoring/loki-config.yaml + servicio loki en docker-compose.yml",
        "qué es": "Base de datos de logs de Grafana.",
        "para qué sirve": "Centraliza logs de contenedores y permite consultarlos con LogQL.",
        "dónde se ve": "http://localhost:3100/ready y /loki/api/v1/query_range.",
    },
    {
        "componente": "Promtail",
        "ubicación": "monitoring/promtail-config.yaml + servicio promtail en docker-compose.yml",
        "qué es": "Agente recolector de logs.",
        "para qué sirve": "Lee logs Docker y los envía a Loki con labels como compose_project y compose_service.",
        "dónde se ve": "docker compose logs promtail.",
    },
    {
        "componente": "system_events",
        "ubicación": "MongoDB hospital.system_events",
        "qué es": "Colección documental de eventos funcionales.",
        "para qué sirve": "Auditar cada ejecución del pipeline por correlation_id/run_id.",
        "dónde se ve": "Mongo Express o mongosh.",
    },
    {
        "componente": "ingestion_rejects",
        "ubicación": "MongoDB hospital.ingestion_rejects",
        "qué es": "Colección de registros rechazados por validación.",
        "para qué sirve": "Demostrar calidad de datos y trazabilidad de errores de entrada.",
        "dónde se ve": "MongoDB: db.ingestion_rejects.find().",
    },
    {
        "componente": "automation",
        "ubicación": "services/automation/main.py",
        "qué es": "Servicio de automatización operativa.",
        "para qué sirve": "Escanea eventos warning/error y crea alertas deduplicadas.",
        "dónde se ve": "docker compose logs automation y MongoDB db.alerts.find().",
    },
    {
        "componente": "alerts",
        "ubicación": "MongoDB hospital.alerts",
        "qué es": "Colección de alertas simuladas.",
        "para qué sirve": "Representa la notificación operativa ante fallos o anomalías.",
        "dónde se ve": "Mongo Express o mongosh.",
    },
])

display(components_df)


# =============================================================================
# 3. Flujo completo
# =============================================================================

md("""
## 2. Flujo completo de logs, eventos y alertas
""")

code_block("text", """
Pipeline ejecuta batch
        |
        |-- logging_setup.py
        |       |
        |       v
        |   stdout JSON del contenedor pipeline
        |       |
        |       v
        |   Promtail recoge logs Docker
        |       |
        |       v
        |   Loki centraliza logs
        |
        |-- events.py
                |
                v
            MongoDB.system_events
                |
                v
            automation escanea warning/error
                |
                v
            MongoDB.alerts
                |
                v
            Dashboard / Mongo Express / revisión operativa
""")

md("""
Hay dos canales complementarios:

1. **Logs técnicos**  
   Salen por stdout del contenedor y se centralizan en Loki.

2. **Eventos de dominio**  
   Se guardan en MongoDB en `system_events` y representan hechos funcionales del pipeline.

Ejemplo:

- Log técnico: `pipeline.run.end` emitido en stdout.
- Evento de dominio: documento en `system_events` con `records_in`, `valid`, `rejected`, `duration_ms`, etc.
""")


# =============================================================================
# 4. Archivos relevantes
# =============================================================================

md("""
## 3. Archivos relevantes del sistema de monitorización
""")

monitoring_files = [
    ROOT / "monitoring" / "loki-config.yaml",
    ROOT / "monitoring" / "promtail-config.yaml",
    ROOT / "services" / "pipeline" / "app" / "logging_setup.py",
    ROOT / "services" / "pipeline" / "app" / "events.py",
    ROOT / "services" / "pipeline" / "config" / "validation_rules.yaml",
    ROOT / "services" / "automation" / "main.py",
    ROOT / "services" / "automation" / "requirements.txt",
    ROOT / "docker-compose.yml",
]

file_rows = []

for path in monitoring_files:
    file_rows.append({
        "archivo": str(path.relative_to(ROOT)) if path.exists() else str(path),
        "existe": path.exists(),
        "líneas": line_count(path) if path.exists() else None,
        "tamaño_kb": file_size_kb(path) if path.exists() else None,
        "símbolos Python detectados": ", ".join(extract_public_symbols(path)[:10]) if path.exists() and path.suffix == ".py" else "",
    })

display(pd.DataFrame(file_rows))


# =============================================================================
# 5. Explicación archivo por archivo
# =============================================================================

md("""
## 4. Explicación archivo por archivo
""")

file_explanation_df = pd.DataFrame([
    {
        "archivo": "monitoring/loki-config.yaml",
        "rol": "Configuración de Loki.",
        "qué hace": "Define cómo arranca Loki, dónde guarda índices/chunks y cómo acepta logs.",
        "por qué importa": "Permite centralizar logs del proyecto y consultarlos por etiquetas.",
    },
    {
        "archivo": "monitoring/promtail-config.yaml",
        "rol": "Configuración de Promtail.",
        "qué hace": "Define cómo descubrir contenedores Docker, qué labels extraer y a qué Loki enviar logs.",
        "por qué importa": "Sin Promtail, Loki estaría levantado pero no recibiría logs de los servicios.",
    },
    {
        "archivo": "services/pipeline/app/logging_setup.py",
        "rol": "Logging JSON.",
        "qué hace": "Configura logs estructurados del pipeline.",
        "por qué importa": "Permite buscar por service, event, correlation_id y level.",
    },
    {
        "archivo": "services/pipeline/app/events.py",
        "rol": "Eventos funcionales.",
        "qué hace": "Inserta eventos en MongoDB `system_events`.",
        "por qué importa": "Da trazabilidad funcional del pipeline, incluso aunque no se mire Loki.",
    },
    {
        "archivo": "services/pipeline/config/validation_rules.yaml",
        "rol": "Reglas de calidad.",
        "qué hace": "Define campos obligatorios, rangos, enums y reglas cruzadas.",
        "por qué importa": "Permite detectar registros incompletos, corruptos o inválidos.",
    },
    {
        "archivo": "services/automation/main.py",
        "rol": "Automatización de alertas.",
        "qué hace": "Escanea `system_events`, detecta warning/error y crea alertas deduplicadas.",
        "por qué importa": "Cumple el requisito de alertas o notificaciones ante fallos de procesamiento.",
    },
    {
        "archivo": "services/automation/requirements.txt",
        "rol": "Dependencias de automation.",
        "qué hace": "Incluye librerías como pymongo.",
        "por qué importa": "Permite construir el servicio de alertas dentro de Docker.",
    },
    {
        "archivo": "docker-compose.yml",
        "rol": "Orquestación.",
        "qué hace": "Levanta loki, promtail, automation, mongodb y el resto de servicios.",
        "por qué importa": "Demuestra que monitorización y alertas forman parte de la infraestructura containerizada.",
    },
])

display(file_explanation_df)


# =============================================================================
# 6. Loki
# =============================================================================

md("""
## 5. Loki: qué es y dónde está

**Loki** es el sistema que centraliza los logs.

En este proyecto:

- se despliega como contenedor `loki`;
- escucha en `localhost:3100`;
- recibe logs enviados por Promtail;
- permite consultas LogQL sobre logs de Docker Compose.

Archivo principal:

`monitoring/loki-config.yaml`
""")

loki_config = ROOT / "monitoring" / "loki-config.yaml"

if loki_config.exists():
    md("### Contenido de `monitoring/loki-config.yaml`")
    code_block("yaml", read_text(loki_config)[:5000])
else:
    md("⚠️ No se encontró `monitoring/loki-config.yaml`.")

loki_block = extract_compose_service_block("loki")

if loki_block:
    md("### Servicio `loki` en `docker-compose.yml`")
    code_block("yaml", loki_block)
else:
    md("⚠️ No se encontró bloque `loki:` en `docker-compose.yml`.")

md("""
### Qué demuestra Loki

Loki demuestra que los logs del pipeline no se quedan solo en la terminal local.

Con Loki se puede consultar:

- logs del servicio `pipeline`;
- logs del servicio `automation`;
- logs de `ml-triage`;
- errores y warnings;
- eventos como `pipeline.run.end`.

Consulta de prueba:

`curl.exe -s http://localhost:3100/ready`

Resultado esperado:

`ready`
""")


# =============================================================================
# 7. Promtail
# =============================================================================

md("""
## 6. Promtail: qué es y para qué sirve

**Promtail** es el agente que recoge los logs de Docker y los envía a Loki.

Relación:

`Docker logs → Promtail → Loki`

Promtail añade labels útiles, por ejemplo:

- `compose_project`;
- `compose_service`;
- `container`;
- `logstream`.

Gracias a esas labels se pueden hacer consultas como:

`{compose_service="pipeline"} |= "pipeline.run.end"`

Archivo principal:

`monitoring/promtail-config.yaml`
""")

promtail_config = ROOT / "monitoring" / "promtail-config.yaml"

if promtail_config.exists():
    md("### Contenido de `monitoring/promtail-config.yaml`")
    code_block("yaml", read_text(promtail_config)[:6000])
else:
    md("⚠️ No se encontró `monitoring/promtail-config.yaml`.")

promtail_block = extract_compose_service_block("promtail")

if promtail_block:
    md("### Servicio `promtail` en `docker-compose.yml`")
    code_block("yaml", promtail_block)
else:
    md("⚠️ No se encontró bloque `promtail:` en `docker-compose.yml`.")

md("""
### Qué demuestra Promtail

Promtail demuestra que el sistema no depende de abrir manualmente `docker compose logs`.

El flujo correcto es:

1. El contenedor `pipeline` escribe logs a stdout.
2. Docker guarda esos logs.
3. Promtail lee los logs Docker.
4. Promtail los etiqueta.
5. Promtail los envía a Loki.
6. Loki permite consultarlos por servicio/proyecto/evento.
""")


# =============================================================================
# 8. system_events
# =============================================================================

md("""
## 7. `system_events`: eventos funcionales del pipeline

`system_events` es una colección de MongoDB.

Guarda eventos de dominio, no solo logs técnicos.

Ejemplos de eventos:

| Evento | Significado |
|---|---|
| `pipeline.run.start` | Inicio del batch |
| `pipeline.file.read` | CSV leído |
| `pipeline.validation.done` | Validación completada |
| `pipeline.rejects.persisted` | Rechazos guardados |
| `pipeline.ml.done` | Predicciones aplicadas |
| `pipeline.aggregates.updated` | Agregados actualizados |
| `pipeline.run.end` | Fin del batch |

Campos habituales:

- `timestamp`;
- `service`;
- `event`;
- `level`;
- `correlation_id`;
- `message`;
- `payload`.

Ejemplo conceptual:
""")

code_block("json", """
{
  "timestamp": "2026-05-15T09:56:05.832Z",
  "service": "pipeline",
  "event": "pipeline.validation.done",
  "level": "warning",
  "correlation_id": "run-20260515-0d2f6e8a",
  "message": "Validación: 1 válidos, 4 rechazados",
  "payload": {
    "valid": 1,
    "rejected": 4
  }
}
""")

md("""
### Diferencia entre logs y `system_events`

| Concepto | Dónde vive | Para qué sirve |
|---|---|---|
| Logs | Loki / Docker stdout | Observabilidad técnica |
| `system_events` | MongoDB | Auditoría funcional del pipeline |

Ambos son útiles. Loki sirve para logs centralizados. MongoDB sirve para consultar eventos de negocio y alimentar alertas/dashboard.
""")


# =============================================================================
# 9. ingestion_rejects
# =============================================================================

md("""
## 8. `ingestion_rejects`: calidad de datos

`ingestion_rejects` es la colección de MongoDB donde se guardan registros rechazados por validación.

Sirve para demostrar:

- detección de campos incompletos;
- detección de enums inválidos;
- detección de rangos imposibles;
- trazabilidad de datos corruptos;
- persistencia de rechazos para revisión.

Ejemplo real esperado con `patients/quality-test-invalid.csv`:
""")

code_block("json", """
{
  "records_in": 5,
  "valid": 1,
  "rejected": 4,
  "rejects_persisted": 4
}
""")

md("Ejemplos de motivos:")

code_block("text", """
missing_field:edad
sexo:enum
intensidad_dolor:max:10
""")


# =============================================================================
# 10. automation
# =============================================================================

md("""
## 9. Servicio `automation`

El servicio `automation` implementa automatización operativa.

Ubicación:

`services/automation/main.py`

Responsabilidades:

1. Conectarse a MongoDB.
2. Leer eventos recientes de `system_events`.
3. Filtrar eventos con `level` igual a `warning` o `error`.
4. Crear alertas deduplicadas en `alerts`.
5. Emitir logs para que Docker/Loki también recojan la actividad.

Este servicio cumple el requisito de:

> Alertas o notificaciones ante fallos en el procesamiento.

En este proyecto la notificación es simulada como documento en MongoDB (`notification_channel = mongo_dashboard`).
""")

automation_main = ROOT / "services" / "automation" / "main.py"

if automation_main.exists():
    md("### Símbolos detectados en `services/automation/main.py`")
    display(pd.DataFrame({"símbolo": extract_public_symbols(automation_main)}))

    md("### Fragmento de `services/automation/main.py`")
    code_block("python", read_text(automation_main)[:7000])
else:
    md("⚠️ No se encontró `services/automation/main.py`.")

automation_block = extract_compose_service_block("automation")

if automation_block:
    md("### Servicio `automation` en `docker-compose.yml`")
    code_block("yaml", automation_block)
else:
    md("⚠️ No se encontró bloque `automation:` en `docker-compose.yml`.")


# =============================================================================
# 11. alerts
# =============================================================================

md("""
## 10. `alerts`: alertas deduplicadas

`alerts` es una colección de MongoDB.

El servicio `automation` crea documentos cuando detecta eventos `warning` o `error`.

Ejemplo conceptual:
""")

code_block("json", """
{
  "dedup_key": "pipeline.validation.done:run-20260515-0d2f6e8a:warning",
  "correlation_id": "run-20260515-0d2f6e8a",
  "severity": "warning",
  "source_service": "pipeline",
  "source_event": "pipeline.validation.done",
  "message": "Validación: 1 válidos, 4 rechazados",
  "payload": {
    "valid": 1,
    "rejected": 4
  },
  "status": "open",
  "notification_channel": "mongo_dashboard",
  "notification_status": "simulated"
}
""")

md("""
### Qué significa deduplicación

La deduplicación evita crear infinitas alertas iguales.

La clave suele combinar:

`evento + correlation_id + nivel`

Por ejemplo:

`pipeline.validation.done:run-20260515-0d2f6e8a:warning`

Si el mismo evento se vuelve a ver, la alerta existente se actualiza en vez de crear otra duplicada.
""")


# =============================================================================
# 12. Colecciones MongoDB
# =============================================================================

md("""
## 11. Colecciones MongoDB relacionadas con monitorización
""")

mongo_collections_df = pd.DataFrame([
    {
        "colección": "system_events",
        "quién escribe": "pipeline / otros servicios",
        "qué guarda": "Eventos funcionales del sistema.",
        "uso": "Auditoría, timeline, alertas, debugging.",
    },
    {
        "colección": "ingestion_rejects",
        "quién escribe": "pipeline",
        "qué guarda": "Registros rechazados por reglas de validación.",
        "uso": "Calidad de datos y revisión de errores de entrada.",
    },
    {
        "colección": "alerts",
        "quién escribe": "automation",
        "qué guarda": "Alertas deduplicadas ante warning/error.",
        "uso": "Notificación simulada y seguimiento operativo.",
    },
    {
        "colección": "aggregates_daily",
        "quién escribe": "pipeline",
        "qué guarda": "Agregados diarios de ejecución.",
        "uso": "Métricas operativas para dashboard.",
    },
    {
        "colección": "predictions_triage",
        "quién escribe": "pipeline / ml-triage",
        "qué guarda": "Predicciones de triaje.",
        "uso": "Mostrar resultado clínico en API/dashboard.",
    },
    {
        "colección": "predictions_disease",
        "quién escribe": "pipeline / ml-triage",
        "qué guarda": "Sospechas de enfermedad.",
        "uso": "Mostrar diagnóstico diferencial orientativo.",
    },
])

display(mongo_collections_df)


# =============================================================================
# 13. Comandos de demostración
# =============================================================================

md("""
## 12. Comandos de demostración

Estos comandos sirven para demostrar la parte de monitorización y calidad de datos.
""")

md("### Ver estado de servicios")

code_block("powershell", """
docker compose ps loki promtail automation mongodb pipeline
""")

md("### Comprobar que Loki está listo")

code_block("powershell", """
curl.exe -s http://localhost:3100/ready
""")

md("### Ver labels que Loki está recibiendo")

code_block("powershell", """
curl.exe -s http://localhost:3100/loki/api/v1/labels
""")

md("### Consultar logs del pipeline en Loki")

code_block("powershell", """
$end = [DateTimeOffset]::UtcNow.ToUnixTimeMilliseconds() * 1000000
$start = [DateTimeOffset]::UtcNow.AddMinutes(-30).ToUnixTimeMilliseconds() * 1000000

$query = '{compose_service="pipeline"} |= "pipeline.run.end"'
$url = "http://localhost:3100/loki/api/v1/query_range?limit=10&start=$start&end=$end&query=$([uri]::EscapeDataString($query))"

curl.exe -s $url
""")

md("### Ejecutar batch inválido para generar warnings")

code_block("powershell", """
docker compose exec pipeline python main.py batch --key "patients/quality-test-invalid.csv"
""")

md("### Consultar eventos warning/error")

code_block("powershell", """
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.system_events.find({level:{`$in:['warning','error']}}).sort({timestamp:-1}).limit(10).pretty()"
""")

md("### Consultar rechazos de calidad")

code_block("powershell", """
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.ingestion_rejects.find().sort({ingested_at:-1}).limit(10).pretty()"
""")

md("### Consultar alertas generadas")

code_block("powershell", """
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.alerts.find().sort({created_at:-1}).limit(10).pretty()"
""")

md("### Ver logs del servicio automation")

code_block("powershell", """
docker compose logs automation --tail=50
""")


# =============================================================================
# 14. Verificación opcional en vivo
# =============================================================================

md("""
## 13. Verificación opcional en vivo

Por defecto no se ejecutan comandos Docker automáticamente.

Cambia `RUN_LIVE_CHECKS = True` si tienes Docker Compose levantado y quieres consultar el estado real desde el notebook.
""")

RUN_LIVE_CHECKS = False

if RUN_LIVE_CHECKS:
    print("=== Estado Loki / Promtail / Automation ===")
    run_cmd("docker compose ps loki promtail automation mongodb pipeline", timeout=60)

    print("\n=== Loki ready ===")
    run_cmd("curl -s http://localhost:3100/ready", timeout=30)

    print("\n=== Labels Loki ===")
    run_cmd("curl -s http://localhost:3100/loki/api/v1/labels", timeout=30)

    print("\n=== Últimos logs de automation ===")
    run_cmd("docker compose logs automation --tail=30", timeout=60)

    print("\n=== Últimos eventos warning/error ===")
    run_cmd(
        "docker compose exec -T mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --quiet --eval \"db.system_events.find({level:{\\$in:['warning','error']}}).sort({timestamp:-1}).limit(5).forEach(printjson)\"",
        timeout=60,
    )

    print("\n=== Últimas alertas ===")
    run_cmd(
        "docker compose exec -T mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --quiet --eval \"db.alerts.find().sort({created_at:-1}).limit(5).forEach(printjson)\"",
        timeout=60,
    )
else:
    md("`RUN_LIVE_CHECKS = False`: no se ejecutan comandos Docker automáticamente.")


# =============================================================================
# 15. Evidencias para defensa
# =============================================================================

md("""
## 14. Evidencias para defensa

Para defender monitorización y calidad de datos, las mejores evidencias son:
""")

evidence_df = pd.DataFrame([
    {
        "evidencia": "docker compose ps loki promtail automation",
        "qué demuestra": "Los servicios de logging y alertas están containerizados.",
    },
    {
        "evidencia": "curl http://localhost:3100/ready",
        "qué demuestra": "Loki está operativo.",
    },
    {
        "evidencia": "curl /loki/api/v1/labels",
        "qué demuestra": "Loki recibe logs etiquetados.",
    },
    {
        "evidencia": "query LogQL {compose_service=\"pipeline\"}",
        "qué demuestra": "Se pueden consultar logs centralizados del pipeline.",
    },
    {
        "evidencia": "db.system_events.find(...)",
        "qué demuestra": "Los eventos funcionales quedan persistidos.",
    },
    {
        "evidencia": "db.ingestion_rejects.find(...)",
        "qué demuestra": "Los registros inválidos se detectan y se guardan.",
    },
    {
        "evidencia": "db.alerts.find(...)",
        "qué demuestra": "El servicio automation crea alertas ante warning/error.",
    },
    {
        "evidencia": "docker compose logs automation",
        "qué demuestra": "El servicio de alertas está funcionando y escaneando eventos.",
    },
])

display(evidence_df)


# =============================================================================
# 16. Resumen final
# =============================================================================

md("""
## 15. Resumen para defensa

La monitorización del proyecto se apoya en dos niveles:

### 1. Logs centralizados

- El pipeline escribe logs JSON.
- Promtail recoge logs de Docker.
- Loki centraliza y permite consultar logs por servicio.

### 2. Eventos funcionales

- El pipeline escribe eventos en MongoDB `system_events`.
- Los rechazos de calidad se guardan en `ingestion_rejects`.
- `automation` revisa eventos `warning/error`.
- `automation` crea alertas deduplicadas en `alerts`.

Con esto se cubren los requisitos de:

- logging centralizado;
- validación de calidad de datos;
- detección de registros incompletos/corruptos;
- alertas ante fallos o anomalías;
- trazabilidad de ejecuciones del pipeline.
""")


ROOT = c:\Users\aripa\Downloads\Practica_Hospital_BACKUP_20260516_145943


# 04 — Monitorización, calidad de datos y alertas

Este notebook explica la parte de monitorización y automatización operativa del proyecto.

El sistema cubre los tres requisitos del enunciado:

| Requisito | Implementación |
|---|---|
| Logging centralizado de procesos del pipeline | Logs JSON del pipeline recogidos por Promtail y enviados a Loki |
| Validación de calidad de datos | `validation_rules.yaml`, eventos `pipeline.validation.done` y colección `ingestion_rejects` |
| Alertas o notificaciones ante fallos | Servicio `automation` que lee `system_events` y crea documentos en `alerts` |

La idea principal es que el sistema no solo procese datos, sino que deje trazabilidad de lo que ha pasado.

## 1. Componentes de monitorización

Esta tabla resume qué papel tiene cada pieza.

,componente,ubicación,qué es,para qué sirve,dónde se ve
0,logging_setup.py,services/pipeline/app/logging_setup.py,Configuración de logs estructurados JSON.,Hace que cada evento del pipeline salga por st...,docker compose logs pipeline; Loki si Promtail...
1,events.py,services/pipeline/app/events.py,Emisor de eventos de dominio.,Guarda eventos funcionales del pipeline en Mon...,MongoDB: db.system_events.find().
2,Loki,monitoring/loki-config.yaml + servicio loki en...,Base de datos de logs de Grafana.,Centraliza logs de contenedores y permite cons...,http://localhost:3100/ready y /loki/api/v1/que...
3,Promtail,monitoring/promtail-config.yaml + servicio pro...,Agente recolector de logs.,Lee logs Docker y los envía a Loki con labels ...,docker compose logs promtail.
4,system_events,MongoDB hospital.system_events,Colección documental de eventos funcionales.,Auditar cada ejecución del pipeline por correl...,Mongo Express o mongosh.
5,ingestion_rejects,MongoDB hospital.ingestion_rejects,Colección de registros rechazados por validación.,Demostrar calidad de datos y trazabilidad de e...,MongoDB: db.ingestion_rejects.find().
6,automation,services/automation/main.py,Servicio de automatización operativa.,Escanea eventos warning/error y crea alertas d...,docker compose logs automation y MongoDB db.al...
7,alerts,MongoDB hospital.alerts,Colección de alertas simuladas.,Representa la notificación operativa ante fall...,Mongo Express o mongosh.


## 2. Flujo completo de logs, eventos y alertas

...

## 13. Verificación opcional en vivo

Por defecto no se ejecutan comandos Docker automáticamente.

Cambia `RUN_LIVE_CHECKS = True` si tienes Docker Compose levantado y quieres consultar el estado real desde el notebook.

`RUN_LIVE_CHECKS = False`: no se ejecutan comandos Docker automáticamente.

## 14. Evidencias para defensa

Para defender monitorización y calidad de datos, las mejores evidencias son:

,evidencia,qué demuestra
0,docker compose ps loki promtail automation,Los servicios de logging y alertas están conta...
1,curl http://localhost:3100/ready,Loki está operativo.
2,curl /loki/api/v1/labels,Loki recibe logs etiquetados.
3,"query LogQL {compose_service=""pipeline""}",Se pueden consultar logs centralizados del pip...
4,db.system_events.find(...),Los eventos funcionales quedan persistidos.
5,db.ingestion_rejects.find(...),Los registros inválidos se detectan y se guardan.
6,db.alerts.find(...),El servicio automation crea alertas ante warni...
7,docker compose logs automation,El servicio de alertas está funcionando y esca...


## 15. Resumen para defensa

La monitorización del proyecto se apoya en dos niveles:

### 1. Logs centralizados

- El pipeline escribe logs JSON.
- Promtail recoge logs de Docker.
- Loki centraliza y permite consultar logs por servicio.

### 2. Eventos funcionales

- El pipeline escribe eventos en MongoDB `system_events`.
- Los rechazos de calidad se guardan en `ingestion_rejects`.
- `automation` revisa eventos `warning/error`.
- `automation` crea alertas deduplicadas en `alerts`.

Con esto se cubren los requisitos de:

- logging centralizado;
- validación de calidad de datos;
- detección de registros incompletos/corruptos;
- alertas ante fallos o anomalías;
- trazabilidad de ejecuciones del pipeline.

## 1. Capas de observabilidad

El proyecto tiene varias capas:

| Capa | Tecnología | Función |
|---|---|---|
| Logs técnicos | stdout JSON | Logs de servicios. |
| Centralización | Loki + Promtail | Recolectar logs Docker. |
| Eventos de dominio | MongoDB `system_events` | Eventos funcionales del pipeline. |
| Alertas | MongoDB `alerts` | Eventos warning/error convertidos en alertas. |
| Dashboard | Streamlit / Mongo Express | Visualización. |

Los logs técnicos se centralizan en Loki mediante Promtail. Los eventos funcionales se guardan en MongoDB en system_events. El servicio automation revisa esos eventos y, cuando detecta warning o error, crea alertas deduplicadas en la colección alerts. Por tanto, Loki cubre logging centralizado y MongoDB cubre trazabilidad funcional y alertas operativas.

In [ ]:
monitoring_files = [
    "monitoring/loki-config.yaml",
    "monitoring/promtail-config.yaml",
    "services/automation/main.py",
    "services/pipeline/app/events.py",
    "services/pipeline/app/logging_setup.py",
]
pd.DataFrame([{"archivo": p, "existe": (ROOT/p).exists()} for p in monitoring_files])


## 2. Loki y Promtail

Loki almacena logs. Promtail recoge logs de Docker.

Validación usada:

```powershell
curl.exe -s http://localhost:3100/ready
```

Consulta LogQL real:

```powershell
$query = '{compose_service="pipeline"} |= "pipeline.run.end"'
$url = "http://localhost:3100/loki/api/v1/query_range?limit=10&start=$start&end=$end&query=$([uri]::EscapeDataString($query))"
curl.exe -s $url
```

Se considera validado cuando Loki devuelve un evento real `pipeline.run.end`.

In [ ]:
print("Comprobar Loki:")
print("curl.exe -s http://localhost:3100/ready")
print()
print("Labels:")
print("curl.exe -s http://localhost:3100/loki/api/v1/labels")


## 3. Eventos de dominio en MongoDB

La colección `system_events` guarda eventos funcionales del pipeline.

Ejemplo:

```json
{
  "event": "pipeline.validation.done",
  "level": "warning",
  "message": "Validación: 1 válidos, 4 rechazados"
}
```

In [ ]:
event_examples = pd.DataFrame([
    {"event": "pipeline.run.start", "level": "info", "count_demo": 1},
    {"event": "pipeline.file.read", "level": "info", "count_demo": 1},
    {"event": "pipeline.validation.done", "level": "warning", "count_demo": 1},
    {"event": "pipeline.rejects.persisted", "level": "warning", "count_demo": 1},
    {"event": "pipeline.run.end", "level": "info", "count_demo": 1},
])
event_examples


In [ ]:
level_counts = event_examples.groupby("level")["count_demo"].sum()
show_bar(level_counts, "Eventos por nivel — ejemplo de ejecución con rechazos", "eventos")


## 4. Automation

El servicio `automation` ejecuta un loop:

```text
cada N segundos
    ↓
lee system_events recientes
    ↓
filtra warning/error
    ↓
crea dedup_key
    ↓
inserta/actualiza alerts
    ↓
loggea nueva alerta
```

Ejemplo de alerta:

```json
{
  "source_event": "pipeline.validation.done",
  "severity": "warning",
  "status": "open",
  "notification_channel": "mongo_dashboard",
  "notification_status": "simulated"
}
```

In [ ]:
alert_examples = pd.DataFrame([
    {"source_event": "pipeline.validation.done", "severity": "warning", "status": "open"},
    {"source_event": "pipeline.rejects.persisted", "severity": "warning", "status": "open"},
])
alert_examples


## 5. Comandos de consulta

Ver alertas:

```powershell
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.alerts.find().sort({created_at:-1}).limit(10).pretty()"
```

Ver eventos warning/error:

```powershell
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.system_events.find({level:{`$in:['warning','error']}}).sort({timestamp:-1}).limit(10).pretty()"
```

Ver logs automation:

```powershell
docker compose logs automation --tail=50
```

## 6. Qué requisito cubre

Esta parte cubre:

- logging centralizado;
- validación de calidad de datos;
- alertas o notificaciones ante fallos del procesamiento;
- trazabilidad con `correlation_id`;
- auditoría de eventos funcionales.